# 02 — Entrenamiento Baseline

Entrenamiento del primer modelo YOLO con dataset Tier 1.

**Objetivo**: Establecer métricas base (mAP, precision, recall) para comparar con entrenamientos posteriores.

In [ ]:
# === Setup para Google Colab ===
import os

if os.getenv('COLAB_RELEASE_TAG'):
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/neurodrive
    !pip install -q ultralytics
    # Verificar GPU
    import torch
    print(f'GPU disponible: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('✅ Entorno Colab configurado')
else:
    print('✅ Entorno local detectado')

In [ ]:
# === Configuración del Entrenamiento ===
# Modificar según necesidades

MODEL = 'yolo11n'           # Modelo base: yolo11n (nano, rápido)
DATASET_YAML = 'configs/datasets/tier1_road_vehicles.yaml'  # Config del dataset
EPOCHS = 50                  # Número de épocas
BATCH_SIZE = 16              # Tamaño de batch (reducir si hay errores de memoria)
IMG_SIZE = 640               # Tamaño de imagen
PATIENCE = 10                # Early stopping
PROJECT = 'results'          # Directorio de resultados
RUN_NAME = 'baseline_v1'     # Nombre de este experimento

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
from ultralytics import YOLO

# Verificar dispositivo
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device != 'cpu':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria GPU: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Cargar Modelo

Cargamos el modelo pre-entrenado de YOLO11 como punto de partida.

In [ ]:
# Cargar modelo pre-entrenado
model = YOLO(f'{MODEL}.pt')
print(f'Modelo cargado: {MODEL}')
print(f'Parámetros: {sum(p.numel() for p in model.model.parameters()):,}')

## Entrenar

Iniciamos el entrenamiento. La celda puede tardar dependiendo del hardware:
- **GPU (Colab T4)**: ~20-30 minutos para 50 épocas
- **CPU**: Mucho más lento, considerar reducir épocas

In [ ]:
# Entrenar modelo
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    project=PROJECT,
    name=RUN_NAME,
    save_period=10,
    plots=True,
    verbose=True,
)

print('\n✅ Entrenamiento completado')

## Resultados del Entrenamiento

Revisamos las métricas finales y las curvas de entrenamiento.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

# Directorio de resultados
results_dir = Path(PROJECT) / RUN_NAME

# Mostrar curvas de entrenamiento
plots = ['results.png', 'confusion_matrix.png', 'P_curve.png', 'R_curve.png']
for plot in plots:
    plot_path = results_dir / plot
    if plot_path.exists():
        print(f'\n📊 {plot}:')
        display(Image(filename=str(plot_path), width=800))

In [ ]:
# Mostrar métricas finales
print('📊 Métricas Finales del Entrenamiento:')
print('=' * 40)

# Cargar mejor modelo para validación
best_model_path = results_dir / 'weights' / 'best.pt'
if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    val_results = best_model.val(data=DATASET_YAML)
    
    print(f'  mAP@50:    {val_results.box.map50:.4f}')
    print(f'  mAP@50-95: {val_results.box.map:.4f}')
    print(f'  Precision: {val_results.box.mp:.4f}')
    print(f'  Recall:    {val_results.box.mr:.4f}')
else:
    print('⚠️ No se encontró best.pt — revisa el directorio de resultados')

## Prueba Rápida

Probamos el modelo entrenado con algunas imágenes.

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Detectar en imágenes de ejemplo
samples_dir = Path('data/samples')
if best_model_path.exists() and samples_dir.exists():
    sample_images = list(samples_dir.glob('*.jpg')) + list(samples_dir.glob('*.png'))
    
    if sample_images:
        for img_path in sample_images[:4]:
            results = best_model(str(img_path), conf=0.25)
            annotated = results[0].plot()
            
            plt.figure(figsize=(10, 6))
            plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
            plt.title(f'{img_path.name} — {len(results[0].boxes)} detecciones')
            plt.axis('off')
            plt.show()
    else:
        print('⚠️ No hay imágenes de ejemplo en data/samples/')
else:
    print('⚠️ Asegúrate de tener el modelo entrenado y algunas imágenes de ejemplo')

## Siguiente Paso

- **03_evaluacion_metricas.ipynb**: Evaluación detallada y comparativa
- Para mejorar: ajustar hiperparámetros en `configs/training/improved.yaml`